# 2. Gene selection under nested cross-validation: the runs

The candidate pool of notebook 1 holds far more genes than the development set has deaths, so a survival model needs a short gene list. This notebook computes and saves every run behind the comparison of ways to choose that list: a shuffled-outcome check, the method comparison and a leakage check. Notebook 3 reads the saved runs and shows the results; it holds no modelling code. With the saved runs present this notebook only reads them back and finishes in seconds. Only the development set is used.

## Settings

The first cell loads the libraries and locates the files. The next two name every fold count, seed, threshold and grid, with its reason beside it. They are saved with the runs, so that notebook 3 can show them.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # folds run in parallel processes; one thread each

import hashlib
import json
import subprocess
import sys
import time
from functools import partial
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.util import Surv


def git(*arguments):
    """The output of a git command run in this repository."""
    completed = subprocess.run(["git", *arguments], capture_output=True, text=True, check=True)
    return completed.stdout.strip()


REPO = Path(git("rev-parse", "--show-toplevel"))
DEVELOPMENT_FILE = REPO / "data" / "processed" / "tcga_development.parquet"  # from notebook 1
DATA_RESULTS = REPO / "results" / "01_data"
RESULTS = REPO / "results" / "02_gene_selection_runs"  # saved runs of this notebook, tracked

In [2]:
OUTER_FOLDS = 5                   # 5-fold cross-validation, the usual choice
OUTER_SEEDS = [1, 2, 3, 4, 5]     # repeated 5 times: 25 held-out folds
INNER_FOLDS = 5                   # tuning folds inside each training part
INNER_SEED = 0                    # one fixed split of every training part
LIST_SIZES = [50, 100, 200, 500, 1000, 2000]  # offered to the ranking methods; first run: to 500
L1_SHARES = [0.25, 0.5, 0.75]     # elastic net mixes between pure ridge (0) and pure LASSO (1)
PATH_LENGTH = 50                  # penalties per path, from the largest down, the library default
PATH_DEPTH = 0.01                 # the smallest penalty is 1% of the largest, the library default
MAX_GENES = 2000                  # longest list a penalty path may offer, the largest list size
RIDGE_GRID = [1.0, 10.0, 100.0, 1000.0, 10000.0]  # ridge of the scoring model; first run: from 100
CLINICAL_RIDGE = 0.001            # numerical safeguard on the clinical coefficients, negligible
LEAKAGE_LIST_SIZE = 100           # list size in the leakage check and the shuffled-outcome check
SHUFFLE_SEEDS = [1, 2, 3, 4, 5]   # shuffled-outcome check: five shuffles of the outcomes
CONFIDENCE = 0.95                 # level of every interval

In [3]:
EXPRESSED_LOG2 = 1.0              # a gene is expressed in a patient at log2(TPM + 1) >= 1, TPM >= 1
EXPRESSED_SHARE = 0.2             # and passes the filter when expressed in 20% of training patients
NEWTON_STEPS = 1000               # guard against an endless loop in the Cox fit
GAIN_LIMIT = 1e-9                 # the Cox fit stops when one more step would improve it by less
EIGEN_FLOOR = 1e-10               # a direction below this share of the largest eigenvalue is empty
DAYS_PER_YEAR = 365.25            # average length of a year, leap years included
UNO_HORIZON = 5 * DAYS_PER_YEAR   # Uno's C is truncated at 5 years
CHECK_SEED = 0                    # seed of the synthetic examples in the checks
N_JOBS = 4                        # outer folds evaluated in parallel, one per physical core
# Indicator columns of the clinical model; the most common level is the reference and gets none.
CLINICAL_LEVELS = {"stage": ["I", "III", "IV", "unknown"], "er": ["negative", "unknown"],
                   "pr": ["negative", "unknown"], "her2": ["positive", "unknown"]}
# The two forms of the scoring model and their key in the saved runs.
FORMS = {"genes only": "genes_only", "genes plus clinical": "genes_plus_clinical"}
INSIDE = "selection inside the training fold"  # the two ways of selecting that are contrasted
ON_ALL = "selection on all patients"

## What is compared

The table names everything that is compared and says whether it looks at patient outcomes when it picks genes. A method that uses outcomes can fit noise, which is why the selection is repeated inside every training part. Four rows are selection methods. Three are references: they appear in the tables and are not candidates for the choice at the end. The last row was planned and not run.

In [4]:
METHODS = {
    "clinical only": {
        "kind": "no genes", "clinical": "clinical", "role": "reference",
        "uses outcomes": "picks no genes",
        "does": "no gene: a Cox model on age, stage, ER, PR and HER2"},
    "clinical only, unknown values replaced": {
        "kind": "no genes", "clinical": "clinical, unknown replaced", "role": "reference",
        "uses outcomes": "picks no genes",
        "does": "as clinical only, with every unknown value replaced by the most common known "
                "level of the training part"},
    "most variable genes": {
        "kind": "ranking by variance", "sizes": LIST_SIZES, "role": "selection method",
        "uses outcomes": "no",
        "does": "ranks genes by the variance of their expression and keeps the top of the ranking"},
    "one-gene Cox ranking": {
        "kind": "ranking by score test", "sizes": LIST_SIZES, "role": "selection method",
        "uses outcomes": "yes",
        "does": "ranks genes by a test of each gene alone in a Cox model and keeps the top of the "
                "ranking"},
    "LASSO-Cox": {
        "kind": "penalised Cox", "l1_shares": [1.0], "role": "selection method",
        "uses outcomes": "yes",
        "does": "fits one Cox model on all genes with a penalty that sets most coefficients to "
                "zero; the remaining genes are the list"},
    "elastic-net Cox": {
        "kind": "penalised Cox", "l1_shares": L1_SHARES, "role": "selection method",
        "uses outcomes": "yes",
        "does": "as LASSO-Cox, with a mixed penalty that lets correlated genes enter together"},
    "all genes": {
        "kind": "no selection", "role": "reference", "uses outcomes": "picks no genes",
        "does": "no selection: the scoring model on every gene that passes the expression filter"}}
NOT_RUN = {
    "the earlier pipeline's selector": {
        "role": "not run", "uses outcomes": "yes, as classes",
        "does": "LASSO on survival classes; planned for the first run, and not run because its "
                "built-in cross-validation exceeded the compute limit"}}
SELECTING_KINDS = ["ranking by variance", "ranking by score test", "penalised Cox"]

In [5]:
def show_table(rows, first_column):
    """Render a dictionary of rows as a Markdown table, which wraps long sentences."""
    columns = list(next(iter(rows.values())))
    header = "| " + first_column + " | " + " | ".join(columns) + " |"
    lines = [header]
    lines.append("|---" * (len(columns) + 1) + "|")
    for name, row in rows.items():
        cells = [name]
        for column in columns:
            cells.append(str(row[column]))
        lines.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(lines)))


SELECTION_METHODS = []
compared = {}
for name, method in METHODS.items():
    if method["role"] == "selection method":
        SELECTION_METHODS.append(name)
for name, method in list(METHODS.items()) + list(NOT_RUN.items()):
    compared[name] = {"what it does": method["does"],
                      "uses patient outcomes to pick genes": method["uses outcomes"],
                      "role": method["role"]}
RESULTS.mkdir(parents=True, exist_ok=True)
compared_table = pd.DataFrame.from_dict(compared, orient="index")
compared_table.to_csv(RESULTS / "what_is_compared.csv")
show_table(compared, "name")

| name | what it does | uses patient outcomes to pick genes | role |
|---|---|---|---|
| clinical only | no gene: a Cox model on age, stage, ER, PR and HER2 | picks no genes | reference |
| clinical only, unknown values replaced | as clinical only, with every unknown value replaced by the most common known level of the training part | picks no genes | reference |
| most variable genes | ranks genes by the variance of their expression and keeps the top of the ranking | no | selection method |
| one-gene Cox ranking | ranks genes by a test of each gene alone in a Cox model and keeps the top of the ranking | yes | selection method |
| LASSO-Cox | fits one Cox model on all genes with a penalty that sets most coefficients to zero; the remaining genes are the list | yes | selection method |
| elastic-net Cox | as LASSO-Cox, with a mixed penalty that lets correlated genes enter together | yes | selection method |
| all genes | no selection: the scoring model on every gene that passes the expression filter | picks no genes | reference |
| the earlier pipeline's selector | LASSO on survival classes; planned for the first run, and not run because its built-in cross-validation exceeded the compute limit | yes, as classes | not run |

## Fixed before running

The protocol was written down and committed before the run. The table is rendered from the settings above, so it cannot drift from the code; the settings are saved beside the runs. Harrell's C, the main metric, is the share of comparable patient pairs that a model ranks in the right order: 0.5 is chance and 1 is a perfect ranking. Uno's C is a variant that corrects for censoring. A gene list is scored by a Cox model refit on it, the scoring model, defined further down. Outer and inner folds are explained under nested cross-validation.

In [6]:
protocol = {}
protocol["Data"] = "the development set of notebook 1 only"
protocol["Outcome"] = "overall survival as time and event"
protocol["Primary metric"] = "Harrell's C on the outer validation folds"
protocol["Secondary metric"] = f"Uno's C truncated at {UNO_HORIZON / DAYS_PER_YEAR:.0f} years"
protocol["Resampling"] = (f"{OUTER_FOLDS} outer folds repeated {len(OUTER_SEEDS)} times (seeds "
                          f"{OUTER_SEEDS[0]} to {OUTER_SEEDS[-1]}), stratified by event; all "
                          f"tuning in {INNER_FOLDS} inner folds of each outer training part")
protocol["Fitted on training parts only"] = ("expression filter, z-scoring, age median, most "
                                             "common clinical level, selection, tuning, the "
                                             "scoring model")
protocol["Scoring model"] = ("ridge Cox refit on the listed genes, in two forms, each tuned on its "
                             "own: genes only (primary) and genes plus clinical")
protocol["Tuning"] = (f"list size in {LIST_SIZES} for the two ranking methods; a path of "
                      f"{PATH_LENGTH} penalties for LASSO-Cox and elastic-net Cox (L1 share in "
                      f"{L1_SHARES}), lists of at most {MAX_GENES} genes; ridge penalty of the "
                      f"scoring model in {RIDGE_GRID}; the highest mean inner-fold Harrell's C "
                      "wins")
protocol["Primary comparison"] = ("elastic-net Cox lists score higher than LASSO-Cox lists, genes "
                                  f"only; a null if the {CONFIDENCE:.0%} interval of the mean "
                                  "paired difference includes 0")
protocol["Secondary comparisons"] = ("genes plus clinical minus clinical only, for each of the "
                                     "four selection methods; a null if the interval includes 0")
protocol["Interval"] = "corrected resampled t for repeated cross-validation"
protocol["Choice rule"] = ("among the four selection methods, highest mean C, genes only; methods "
                           "whose interval against it includes 0 are tied; among tied methods "
                           "prefer one that uses no outcome, then the higher Jaccard index")
protocol["Leakage check"] = (f"{ON_ALL} against {INSIDE}, for one-gene Cox ranking at "
                             f"{LEAKAGE_LIST_SIZE} genes, LASSO-Cox and elastic-net Cox")
protocol["Shuffled-outcome check"] = ("the same contrast for one-gene Cox ranking on "
                                      f"{len(SHUFFLE_SEEDS)} shuffles of the outcomes, run first; "
                                      f"with {INSIDE} the interval must include 0.50")
protocol["After the run"] = "the grids are not changed again, whatever the grid-edge table shows"
protocol_rows = {}
for item, choice in protocol.items():
    protocol_rows[item] = {"fixed choice": choice}
show_table(protocol_rows, "item")

| item | fixed choice |
|---|---|
| Data | the development set of notebook 1 only |
| Outcome | overall survival as time and event |
| Primary metric | Harrell's C on the outer validation folds |
| Secondary metric | Uno's C truncated at 5 years |
| Resampling | 5 outer folds repeated 5 times (seeds 1 to 5), stratified by event; all tuning in 5 inner folds of each outer training part |
| Fitted on training parts only | expression filter, z-scoring, age median, most common clinical level, selection, tuning, the scoring model |
| Scoring model | ridge Cox refit on the listed genes, in two forms, each tuned on its own: genes only (primary) and genes plus clinical |
| Tuning | list size in [50, 100, 200, 500, 1000, 2000] for the two ranking methods; a path of 50 penalties for LASSO-Cox and elastic-net Cox (L1 share in [0.25, 0.5, 0.75]), lists of at most 2000 genes; ridge penalty of the scoring model in [1.0, 10.0, 100.0, 1000.0, 10000.0]; the highest mean inner-fold Harrell's C wins |
| Primary comparison | elastic-net Cox lists score higher than LASSO-Cox lists, genes only; a null if the 95% interval of the mean paired difference includes 0 |
| Secondary comparisons | genes plus clinical minus clinical only, for each of the four selection methods; a null if the interval includes 0 |
| Interval | corrected resampled t for repeated cross-validation |
| Choice rule | among the four selection methods, highest mean C, genes only; methods whose interval against it includes 0 are tied; among tied methods prefer one that uses no outcome, then the higher Jaccard index |
| Leakage check | selection on all patients against selection inside the training fold, for one-gene Cox ranking at 100 genes, LASSO-Cox and elastic-net Cox |
| Shuffled-outcome check | the same contrast for one-gene Cox ranking on 5 shuffles of the outcomes, run first; with selection inside the training fold the interval must include 0.50 |
| After the run | the grids are not changed again, whatever the grid-edge table shows |

In [7]:
settings = {"outer folds": OUTER_FOLDS, "outer seeds": OUTER_SEEDS, "inner folds": INNER_FOLDS,
            "inner seed": INNER_SEED, "list sizes": LIST_SIZES, "L1 shares": L1_SHARES,
            "path length": PATH_LENGTH, "path depth": PATH_DEPTH, "longest list": MAX_GENES,
            "ridge grid": RIDGE_GRID, "clinical ridge": CLINICAL_RIDGE,
            "leakage list size": LEAKAGE_LIST_SIZE, "shuffle seeds": SHUFFLE_SEEDS,
            "confidence": CONFIDENCE, "expressed log2": EXPRESSED_LOG2,
            "expressed share": EXPRESSED_SHARE, "Uno horizon (days)": UNO_HORIZON,
            "clinical levels": CLINICAL_LEVELS, "forms": FORMS,
            "arms": {"inside": INSIDE, "on all": ON_ALL}, "selection methods": SELECTION_METHODS}
(RESULTS / "settings.json").write_text(json.dumps(settings, indent=1))

1067

## The development set

We load the development file and compare its checksum with the one notebook 1 recorded, so the experiment provably runs on the locked split. The data are turned into plain arrays: the expression matrix, the outcome, the age, and the level of each clinical covariate.

In [8]:
recorded = json.loads((DATA_RESULTS / "split.json").read_text())["tcga_development"]
file_sha256 = hashlib.sha256(DEVELOPMENT_FILE.read_bytes()).hexdigest()
development = pd.read_parquet(DEVELOPMENT_FILE)
gene_ids = []
for column in development.columns:
    if column.startswith("ENSG"):
        gene_ids.append(column)
data = {"expression": development[gene_ids].to_numpy(),
        "time": development["os_days"].to_numpy(dtype=float),
        "event": development["os_event"].to_numpy(dtype=bool),
        "age": development["age"].to_numpy(dtype=float)}
for name in CLINICAL_LEVELS:
    data[name] = development[name].to_numpy(dtype=str)
deaths = data["event"].sum()
display(Markdown(f"Development set: {len(development)} patients, {deaths} deaths, "
                 f"{len(gene_ids)} candidate genes."))

Development set: 851 patients, 120 deaths, 19211 candidate genes.

In [9]:
# The experiment is only valid on the split that notebook 1 locked.
assert file_sha256 == recorded["file sha256"], "the development file differs from notebook 1's"
assert len(development) == recorded["patients"], "the number of patients differs from notebook 1's"
assert deaths == recorded["deaths"], "the number of deaths differs from notebook 1's"

## Preparing a fold

Every step that learns from data is fitted on the training part of a fold and then applied to the held-out part: the expression filter, the scaling of each gene to mean 0 and variance 1 (z-scoring), the median used for a missing age, and the most common level used when unknown clinical values are replaced. Using held-out patients in any of these steps is called leakage: the evaluation would then test a model on patients that helped to build it.

In [10]:
def clinical_columns(data, train, replace_unknown):
    """Age and indicator columns per clinical level, every statistic from the training part."""
    age = np.where(np.isnan(data["age"]), np.nanmedian(data["age"][train]), data["age"])
    columns = [(age - age[train].mean()) / age[train].std()]
    for name, levels in CLINICAL_LEVELS.items():
        values = data[name]
        if replace_unknown:
            known = values[train][values[train] != "unknown"]
            most_common = pd.Series(known).mode().iloc[0]
            values = np.where(values == "unknown", most_common, values)
        for level in levels:
            if replace_unknown and level == "unknown":
                continue
            columns.append(values == level)
    return np.column_stack(columns).astype(float)


def prepare_fold(data, train, valid):
    """Fit the filter, the scaling and the clinical columns on the training part; apply to both."""
    log_train = data["expression"][train]
    mean = log_train.mean(axis=0)
    sd = log_train.std(axis=0)
    sd[sd == 0] = 1.0  # a constant gene never passes the filter; this only avoids a division by 0
    expressed = (log_train >= EXPRESSED_LOG2).mean(axis=0) >= EXPRESSED_SHARE
    fold = {"genes_train": (log_train - mean) / sd,
            "genes_valid": (data["expression"][valid] - mean) / sd,
            "expressed": np.flatnonzero(expressed),
            "variance": log_train.var(axis=0),
            "y_train": Surv.from_arrays(data["event"][train], data["time"][train]),
            "y_valid": Surv.from_arrays(data["event"][valid], data["time"][valid])}
    for name, replace_unknown in [("clinical", False), ("clinical, unknown replaced", True)]:
        columns = clinical_columns(data, train, replace_unknown)
        fold[name] = (columns[train], columns[valid])
    return fold

In [11]:
# Leakage check: nothing fitted in prepare_fold may depend on the held-out patients, so replacing
# them by noise must change nothing.
rng = np.random.default_rng(CHECK_SEED)
half = len(data["time"]) // 2
train = np.arange(half)
valid = np.arange(half, len(data["time"]))
noisy = dict(data)
noisy["expression"] = data["expression"].copy()
noisy["age"] = data["age"].copy()
noisy["stage"] = data["stage"].copy()
noisy["expression"][valid] = rng.normal(size=(len(valid), len(gene_ids)))
noisy["age"][valid] = rng.normal(size=len(valid))
noisy["stage"][valid] = "IV"
original = prepare_fold(data, train, valid)
scrambled = prepare_fold(noisy, train, valid)
for key in ["genes_train", "expressed", "variance"]:
    assert np.array_equal(original[key], scrambled[key]), "prepare_fold uses the held-out patients"
for key in ["clinical", "clinical, unknown replaced"]:
    assert np.array_equal(original[key][0], scrambled[key][0]), "clinical columns use held-out data"

## The scoring model

A gene list is scored by a Cox regression refit on it, the scoring model. Cox regression is the standard model for censored survival times: every patient gets a risk score, a weighted sum of the covariates, and a higher score means a higher hazard, the momentary risk of death. The refit uses a ridge penalty, which shrinks all coefficients towards zero, and is scored on patients it has not seen. It has two forms: genes only, and genes plus clinical, where the clinical covariates join the genes, practically unpenalised. An empty list scores 0.5.

In [12]:
def scoring_model_risk(fold, columns, ridge, form, clinical="clinical"):
    """Fit the ridge Cox model on the columns and return the held-out patients' risk scores."""
    train, valid = columns
    penalty = np.full(train.shape[1], ridge)
    if form == "genes plus clinical":
        clinical_train, clinical_valid = fold[clinical]
        train = np.hstack([clinical_train, train])
        valid = np.hstack([clinical_valid, valid])
        clinical_penalty = np.full(clinical_train.shape[1], CLINICAL_RIDGE)
        penalty = np.concatenate([clinical_penalty, penalty])
    if train.shape[1] == 0:
        return np.zeros(len(valid))  # no column: every patient gets the same score, C = 0.5
    y = fold["y_train"]
    return valid @ fit_ridge_cox(train, y["time"], y["event"], penalty)


def harrell_c(fold, risk):
    """Harrell's C of the risk scores on the held-out part."""
    y = fold["y_valid"]
    return concordance_index_censored(y["event"], y["time"], risk)[0]


def uno_c(fold, risk):
    """Uno's C of the risk scores on the held-out part, truncated at the horizon."""
    return concordance_index_ipcw(fold["y_train"], fold["y_valid"], risk, tau=UNO_HORIZON)[0]

## Numerical routines

The next four functions replace library calls that would be too slow for this run: the Cox fit of the scoring model, a shortcut for lists with more genes than patients, the one-gene score test, and the penalty paths, where the inner folds must refit at the penalties of the outer training part. Each gives the same answer as the library, and the assert cells after them check it on synthetic examples. A reader may skip this section.

The Cox likelihood takes each death and asks what chance the model gives that patient, among all still at risk, of being the one who dies. Newton's method improves the coefficients from the slope (gradient) and curvature (Hessian) of this likelihood, halving any step that would worsen the fit, and stops when one more step would gain less than the limit set above.

In [13]:
def risk_set_chances(beta, X, outside, died, penalty):
    """Per death, each patient's chance of being the one who dies; and the penalised loss."""
    score = X @ beta
    masked = score + outside  # minus infinity gives a patient outside the risk set the weight 0
    top = masked.max(axis=1, keepdims=True)  # taken out of the exponentials against overflow
    weight = np.exp(masked - top)
    total = weight.sum(axis=1, keepdims=True)
    loss = (top + np.log(total)).sum() - died @ score + 0.5 * penalty @ beta ** 2
    return weight / total, loss


def fit_ridge_cox(X, time, event, penalty):
    """Coefficients of a Cox model with one ridge penalty per column (ties share a risk set)."""
    died = event.astype(float)
    # One row per death: 0 for a patient still at risk at that time, minus infinity otherwise.
    outside = np.where(time[None, :] >= time[event][:, None], 0.0, -np.inf)
    beta = np.zeros(X.shape[1])
    chance, loss = risk_set_chances(beta, X, outside, died, penalty)
    for _ in range(NEWTON_STEPS):
        mean = chance @ X  # per death, the expected covariates of the one who dies
        expected = chance.sum(axis=0)  # per patient, the expected number of deaths
        gradient = X.T @ (expected - died) + penalty * beta
        weighted = X * np.sqrt(expected)[:, None]
        hessian = weighted.T @ weighted - mean.T @ mean + np.diag(penalty)
        step = np.linalg.solve(hessian, gradient)
        if gradient @ step / 2 < GAIN_LIMIT:  # the improvement that the next step promises
            return beta - step
        new_chance, new_loss = risk_set_chances(beta - step, X, outside, died, penalty)
        while new_loss > loss:
            step = step / 2
            new_chance, new_loss = risk_set_chances(beta - step, X, outside, died, penalty)
        beta, chance, loss = beta - step, new_chance, new_loss
    raise RuntimeError("the Cox fit did not converge")

A list can hold more genes than there are patients. The fit then needs a matrix with one row and one column per gene, which is slow for long lists. With a ridge penalty the same model can be written on far fewer columns: the patients-by-patients matrix of gene products decides the fit, and its eigenvectors give at most one column per patient with exactly the same risk scores.

In [14]:
def gene_columns(fold, genes):
    """Z-scored columns of the listed genes for both parts; a long list becomes fewer columns."""
    train = fold["genes_train"][:, genes]
    valid = fold["genes_valid"][:, genes]
    if train.shape[1] <= train.shape[0]:
        return train, valid
    values, vectors = np.linalg.eigh(train @ train.T)
    keep = values > values.max() * EIGEN_FLOOR
    scale = np.sqrt(values[keep])
    short_train = vectors[:, keep] * scale
    short_valid = (valid @ train.T) @ (vectors[:, keep] / scale)
    return short_train, short_valid

In [15]:
# The fit must agree with the Cox model of the library, with two penalty levels and tied times.
# Synthetic example: 200 patients, whole-number times so that ties occur; first 12 columns, then
# 300 columns, more than patients. The library is asked for a tight fit, so that the two can be
# compared to six decimals.
library_tolerance = 1e-12
agreement = 1e-6
rng = np.random.default_rng(CHECK_SEED)
x = rng.normal(size=(200, 300))
t = rng.integers(1, 40, size=200).astype(float)
e = rng.random(200) < 0.5
y = Surv.from_arrays(e, t)
penalty = np.concatenate([np.full(4, CLINICAL_RIDGE), np.full(8, RIDGE_GRID[0])])
library = CoxPHSurvivalAnalysis(alpha=penalty, tol=library_tolerance).fit(x[:, :12], y).coef_
ours = fit_ridge_cox(x[:, :12], t, e, penalty)
assert np.allclose(ours, library, atol=agreement), "the fit disagrees with the library's Cox model"
example = {"genes_train": x, "genes_valid": x}
short_train, short_valid = gene_columns(example, np.arange(300))
assert short_train.shape[1] <= 200, "the shortcut must not use more columns than patients"
for ridge in RIDGE_GRID:
    short_beta = fit_ridge_cox(short_train, t, e, np.full(short_train.shape[1], ridge))
    shortcut_risk = short_valid @ short_beta
    direct_risk = x @ fit_ridge_cox(x, t, e, np.full(300, ridge))
    assert np.allclose(shortcut_risk, direct_risk, atol=agreement), "the shortcut changes the risks"
    if ridge == RIDGE_GRID[2]:
        library_fit = CoxPHSurvivalAnalysis(alpha=ridge, tol=library_tolerance).fit(x, y)
        library_risk = x @ library_fit.coef_
        assert np.allclose(shortcut_risk, library_risk, atol=agreement), "shortcut versus library"

The one-gene Cox ranking uses a score test: for each gene alone, it asks how strongly expression and hazard go together in a one-gene Cox model. The library has no fast version of this test, so the next cell computes it for all genes at once; the cell after it checks the result against the textbook formula on a small synthetic example.

In [16]:
def cox_score_statistics(Z, time, event):
    """Score test statistic of every column of Z in its own one-gene Cox model (ties share)."""
    order = np.argsort(-time, kind="stable")  # longest time first: a running sum is then a risk set
    Z = Z[order]
    time = time[order]
    event = event[order]
    at_risk = np.searchsorted(-time, -time, side="right")[event]  # size of the risk set per death
    mean = np.cumsum(Z, axis=0)[at_risk - 1] / at_risk[:, None]
    mean_of_squares = np.cumsum(Z ** 2, axis=0)[at_risk - 1] / at_risk[:, None]
    observed_minus_expected = (Z[event] - mean).sum(axis=0)
    return observed_minus_expected / np.sqrt((mean_of_squares - mean ** 2).sum(axis=0))

In [17]:
# The fast score test must agree with the textbook formula computed death by death.
# Synthetic example: 40 patients, 3 genes, whole-number times up to 14 so that ties occur, about
# half of the patients die.
rng = np.random.default_rng(CHECK_SEED)
z = rng.normal(size=(40, 3))
t = rng.integers(1, 15, size=40).astype(float)
e = rng.random(40) < 0.5
slow = np.zeros(3)
spread = np.zeros(3)
for death in np.flatnonzero(e):
    risk_set = z[t >= t[death]]
    slow += z[death] - risk_set.mean(axis=0)
    spread += risk_set.var(axis=0)
fast = cox_score_statistics(z, t, e)
assert np.allclose(fast, slow / np.sqrt(spread)), "the fast score test disagrees with the formula"

LASSO-Cox is a Cox model on all genes with a penalty that sets most coefficients exactly to zero; the genes left with a non-zero coefficient are the list. The elastic net mixes that penalty with a ridge penalty and lets correlated genes enter together; the L1 share says how much of the mix is the LASSO penalty. Both are fitted along a path of penalties by the library, and each penalty yields one candidate list. Inner folds refit at the penalties of the outer training part.

In [18]:
def path_lists(method, fold, settings=None):
    """Every list a penalty path offers on one training part; inner folds refit at set penalties."""
    expressed = fold["expressed"]
    Z = fold["genes_train"][:, expressed]
    lists = {}
    for share in method["l1_shares"]:
        penalties = None
        if settings is not None:
            penalties = []
            for given_share, penalty in settings:
                if given_share == share:
                    penalties.append(penalty)
        path = CoxnetSurvivalAnalysis(l1_ratio=share, n_alphas=PATH_LENGTH,
                                      alpha_min_ratio=PATH_DEPTH, alphas=penalties)
        path.fit(Z, fold["y_train"])
        for column, penalty in enumerate(path.alphas_):
            genes = expressed[np.flatnonzero(path.coef_[:, column])]
            if settings is None and len(genes) > MAX_GENES:
                continue  # the length limit is applied once, on the outer training part
            lists[(share, float(penalty))] = genes
    return lists

## The selection methods

The two ranking methods order the expressed genes, by variance or by the score test, and offer the top of the ranking at each list size. The function below returns every list a method offers on one training part, keyed by its setting; a fixed list and the all-genes reference offer exactly one.

In [19]:
def ranking_lists(method, fold):
    """Nested lists from a ranking of the expressed genes, by variance or by the score test."""
    expressed = fold["expressed"]
    if method["kind"] == "ranking by variance":
        score = fold["variance"][expressed]
    else:
        Z = fold["genes_train"][:, expressed]
        y = fold["y_train"]
        score = np.abs(cox_score_statistics(Z, y["time"], y["event"]))
    ranking = expressed[np.argsort(-score, kind="stable")]
    lists = {}
    for size in method["sizes"]:
        lists[size] = ranking[:size]
    return lists


def candidate_lists(method, fold, settings=None):
    """All gene lists a method offers on one training part, keyed by setting."""
    if method["kind"] == "fixed list":
        return {"fixed": np.asarray(method["genes"], dtype=int)}
    if method["kind"] == "no selection":
        return {"all expressed genes": fold["expressed"]}
    if method["kind"].startswith("ranking"):
        return ranking_lists(method, fold)
    return path_lists(method, fold, settings)

## Nested cross-validation

Cross-validation holds out one fold of patients, fits on the rest and scores on the held-out fold. In nested cross-validation every choice is itself made by an inner cross-validation that only sees the training part: here the setting of the method and the ridge penalty of the scoring model, chosen separately for each of its two forms. The selection is redone on every inner training part.

In [20]:
def best_pair(scores):
    """The (setting, ridge) pair with the highest mean inner C; a tie goes to the first in order."""
    means = {}
    for pair, values in scores.items():
        means[pair] = np.mean(values)
    return max(means, key=means.get)


def tune(method, data, train, outer):
    """Inner folds of the training part choose the setting and the ridge for each form."""
    lists = candidate_lists(method, outer)
    scores = {}
    for form in FORMS:
        scores[form] = {}
        for setting in lists:
            for ridge in RIDGE_GRID:
                scores[form][(setting, ridge)] = []
    splitter = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=INNER_SEED)
    for inner_train, inner_valid in splitter.split(train, data["event"][train]):
        inner = prepare_fold(data, train[inner_train], train[inner_valid])
        inner_lists = candidate_lists(method, inner, settings=list(lists))
        for setting in lists:
            columns = gene_columns(inner, inner_lists[setting])
            for ridge in RIDGE_GRID:
                for form in FORMS:
                    risk = scoring_model_risk(inner, columns, ridge, form)
                    scores[form][(setting, ridge)].append(harrell_c(inner, risk))
    chosen = {}
    for form in FORMS:
        chosen[form] = best_pair(scores[form])
    return lists, chosen

The held-out fold of the outer loop is used once, to score the lists chosen without it. One record per fold keeps, for each form of the scoring model, both C statistics, the chosen setting and ridge, the list length, whether the list was the shortest or the longest on offer, and the list itself.

In [21]:
def evaluate_fold(method, data, train, valid):
    """One outer fold: tune on the training part only, then score each form on the held-out part."""
    outer = prepare_fold(data, train, valid)
    if method["kind"] == "no genes":
        lists = {"none": np.array([], dtype=int)}
        chosen = {"genes only": ("none", 0.0), "genes plus clinical": ("none", 0.0)}
    else:
        lists, chosen = tune(method, data, train, outer)
    lengths = []
    for genes in lists.values():
        lengths.append(len(genes))
    record = {}
    for form, key in FORMS.items():
        setting, ridge = chosen[form]
        genes = lists[setting]
        columns = gene_columns(outer, genes)
        clinical = method.get("clinical", "clinical")
        risk = scoring_model_risk(outer, columns, ridge, form, clinical)
        record["harrell_" + key] = harrell_c(outer, risk)
        record["uno_" + key] = uno_c(outer, risk)
        record["setting_" + key] = str(setting)
        record["ridge_" + key] = ridge
        record["list_length_" + key] = len(genes)
        record["shortest_list_" + key] = len(genes) == min(lengths)
        record["longest_list_" + key] = len(genes) == max(lengths)
        record["selected_" + key] = genes
    return record

## Running and saving

The outer folds of a run are evaluated in parallel. Every finished run is saved under `results/02_gene_selection_runs/` with the commit that produced it and its running time; a later execution reads the saved file, and deleting a file makes the notebook compute it again. Chosen lists are saved by Ensembl ID for the four selection methods.

In [22]:
def outer_folds(data, seeds):
    """The outer folds of every repeat: repeat seed, fold index, training rows, held-out rows."""
    folds = []
    for seed in seeds:
        splitter = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=seed)
        for index, (train, valid) in enumerate(splitter.split(data["event"], data["event"])):
            folds.append((seed, index, train, valid))
    return folds


def with_gene_ids(record, keeps_lists):
    """A record for saving: chosen lists as Ensembl IDs, or dropped when there was no choice."""
    saved_record = {}
    for name, value in record.items():
        if not name.startswith("selected"):
            saved_record[name] = value
        elif keeps_lists:
            saved_record[name] = [gene_ids[gene] for gene in value]
    return saved_record


def run_method(method, data, seeds):
    """Evaluate a method on every outer fold of every repeat, several folds at a time."""
    folds = outer_folds(data, seeds)
    tasks = []
    for _, _, train, valid in folds:
        tasks.append(delayed(evaluate_fold)(method, data, train, valid))
    records = Parallel(n_jobs=N_JOBS)(tasks)
    keeps_lists = method["kind"] in SELECTING_KINDS
    rows = []
    for (seed, index, _, _), record in zip(folds, records):
        row = {"repeat": seed, "fold": index}
        row.update(with_gene_ids(record, keeps_lists))
        rows.append(row)
    return rows

In [23]:
def file_name(*name_parts):
    """A saved run is named after what it is, in lower case with underscores."""
    name = "_".join(name_parts).lower()
    return name.replace("-", "_").replace(",", "").replace(" ", "_") + ".json"


def saved(*name_parts, compute):
    """Return the saved result of a run, or compute it, save it and return it."""
    path = RESULTS / file_name(*name_parts)
    if not path.exists():
        started = time.time()
        result = compute()
        saved_run = {"git commit": git("rev-parse", "HEAD"),
                     "seconds": round(time.time() - started), "result": result}
        path.write_text(json.dumps(saved_run, indent=1))
    return json.loads(path.read_text())["result"]


def selected_on_all(method, data):
    """The whole selection, tuning included, once on all patients; the genes-only list as IDs."""
    everyone = np.arange(len(data["time"]))
    lists, chosen = tune(method, data, everyone, prepare_fold(data, everyone, everyone))
    setting = chosen["genes only"][0]
    return [gene_ids[gene] for gene in lists[setting]]


def fixed_list(genes):
    """A method that always offers the given Ensembl IDs, so only the scoring model is tuned."""
    return {"kind": "fixed list", "genes": pd.Index(gene_ids).get_indexer(genes)}

## Shuffled-outcome check

Before any real outcome is modelled we test the machinery itself. We shuffle the outcomes across patients, which destroys every true link between genes and survival, and run the one-gene Cox ranking both ways: with selection inside the training fold, and with selection on all patients before the cross-validation. A correct pipeline must score about 0.5 with selection inside the training fold; the assert stops the notebook otherwise.

In [24]:
def shuffled_outcome_check():
    """Both arms of the leakage check on shuffled outcomes, one 5-fold split per shuffle."""
    method = {"kind": "ranking by score test", "sizes": [LEAKAGE_LIST_SIZE]}
    rows = []
    for seed in SHUFFLE_SEEDS:
        order = np.random.default_rng(seed).permutation(len(data["time"]))
        shuffled = dict(data)
        shuffled["time"] = data["time"][order]
        shuffled["event"] = data["event"][order]
        arms = [(INSIDE, method), (ON_ALL, fixed_list(selected_on_all(method, shuffled)))]
        for arm, arm_method in arms:
            for record in run_method(arm_method, shuffled, [seed]):
                row = {"shuffle": seed, "arm": arm}
                row.update(record)
                rows.append(row)
    return rows


def shuffle_means(rows, arm):
    """The mean C of each shuffle for one arm."""
    table = pd.DataFrame(rows)
    table = table[table["arm"] == arm]
    return table.groupby("shuffle")["harrell_genes_only"].mean()


def t_interval(values):
    """Mean and t interval of a few independent values."""
    values = np.asarray(values, dtype=float)
    spread = values.std(ddof=1) / np.sqrt(len(values))
    half = stats.t.ppf((1 + CONFIDENCE) / 2, len(values) - 1) * spread
    return values.mean(), values.mean() - half, values.mean() + half


shuffled_rows = saved("shuffled-outcome check", compute=shuffled_outcome_check)
inside_mean, inside_low, inside_high = t_interval(shuffle_means(shuffled_rows, INSIDE))
on_all_mean, on_all_low, on_all_high = t_interval(shuffle_means(shuffled_rows, ON_ALL))
display(Markdown(f"On shuffled outcomes, {INSIDE} scores {inside_mean:.3f} ({inside_low:.3f} to "
                 f"{inside_high:.3f}) and {ON_ALL} scores {on_all_mean:.3f} ({on_all_low:.3f} to "
                 f"{on_all_high:.3f})."))

On shuffled outcomes, selection inside the training fold scores 0.524 (0.475 to 0.573) and selection on all patients scores 0.665 (0.621 to 0.709).

In [25]:
# If selection inside the training fold finds signal in shuffled outcomes, the pipeline leaks and
# no result below could be trusted.
assert inside_low <= 0.5 <= inside_high, "the code leaks: C away from 0.50 on shuffled outcomes"

## Method comparison

Each row of the table of what is compared is now run on the real outcomes of the development set, on the same outer folds. The runs are saved one by one; the table lists them with their running time.

In [26]:
runs = {}
for name, method in METHODS.items():
    compute = partial(run_method, method, data, OUTER_SEEDS)
    runs[name] = saved("method comparison", name, compute=compute)

## Leakage check

Selection on all patients means: the selection runs once on all development patients, tuning included, and then only the scoring model is cross-validated on the same outer folds. For the one-gene Cox ranking both ways use lists of the fixed length set above, so its nested run is computed here; for LASSO-Cox and elastic-net Cox the nested runs of the method comparison serve.

In [27]:
RANKING = "one-gene Cox ranking"
LEAKAGE_METHODS = {RANKING: {"kind": "ranking by score test", "sizes": [LEAKAGE_LIST_SIZE]},
                   "LASSO-Cox": METHODS["LASSO-Cox"], "elastic-net Cox": METHODS["elastic-net Cox"]}
lists_on_all = {}
on_all = {}
for name, method in LEAKAGE_METHODS.items():
    compute_list = partial(selected_on_all, method, data)
    genes = saved("leakage check", name, "list selected on all patients", compute=compute_list)
    lists_on_all[name] = genes
    compute_run = partial(run_method, fixed_list(genes), data, OUTER_SEEDS)
    on_all[name] = saved("leakage check", name, ON_ALL, compute=compute_run)
compute_inside = partial(run_method, LEAKAGE_METHODS[RANKING], data, OUTER_SEEDS)
inside_ranking = saved("leakage check", RANKING, INSIDE, compute=compute_inside)

## Run record

The last block records the git commit of this execution, the package versions, the seeds, the checksum of the input file, and for each saved run the commit that computed it and its running time.

In [28]:
saved_runs = {}
for path in sorted(RESULTS.glob("*.json")):
    if path.name in ["run_record.json", "settings.json"]:
        continue
    content = json.loads(path.read_text())
    saved_runs[path.stem] = {"git commit": content["git commit"], "seconds": content["seconds"]}
packages = {}
for name in ["numpy", "scipy", "pandas", "pyarrow", "scikit-learn", "scikit-survival", "joblib"]:
    packages[name] = version(name)
changed = git("status", "--porcelain", "--untracked-files=no").splitlines()
run_record = {"git commit": git("rev-parse", "HEAD"),
              "tracked files changed since that commit": changed,
              "python": sys.version.split()[0],
              "packages": packages,
              "seeds": {"outer folds": OUTER_SEEDS, "inner folds": INNER_SEED,
                        "shuffles": SHUFFLE_SEEDS, "checks": CHECK_SEED},
              "input file (sha256)": {"tcga_development.parquet": file_sha256},
              "saved runs": saved_runs}
(RESULTS / "run_record.json").write_text(json.dumps(run_record, indent=1))
for key, value in run_record.items():
    if key != "saved runs":
        print(key + ":", json.dumps(value))
pd.DataFrame.from_dict(saved_runs, orient="index")

git commit: "1df98b1f78cb6c9e9def4bf8f787ad70756b4602"
tracked files changed since that commit: ["M notebooks/01_data.ipynb", " M results/01_data/run_record.json"]
python: "3.12.14"
packages: {"numpy": "2.5.3", "scipy": "1.18.1", "pandas": "3.0.6", "pyarrow": "25.0.1", "scikit-learn": "1.9.1", "scikit-survival": "0.28.0", "joblib": "1.6.0"}
seeds: {"outer folds": [1, 2, 3, 4, 5], "inner folds": 0, "shuffles": [1, 2, 3, 4, 5], "checks": 0}
input file (sha256): {"tcga_development.parquet": "bcbba03ca8e969ba807297f8b0fce1ba464b957ff124b6ff32f7141beaeda146"}


,git commit,seconds
leakage_check_elastic_net_cox_list_selected_on_all_patients,c88e1fb267a4aed17d3779cab916445229089304,750
leakage_check_elastic_net_cox_selection_on_all_patients,c88e1fb267a4aed17d3779cab916445229089304,127
leakage_check_lasso_cox_list_selected_on_all_patients,c88e1fb267a4aed17d3779cab916445229089304,91
leakage_check_lasso_cox_selection_on_all_patients,c88e1fb267a4aed17d3779cab916445229089304,31
leakage_check_one_gene_cox_ranking_list_selected_on_all_patients,c88e1fb267a4aed17d3779cab916445229089304,4
leakage_check_one_gene_cox_ranking_selection_inside_the_training_fold,c88e1fb267a4aed17d3779cab916445229089304,48
leakage_check_one_gene_cox_ranking_selection_on_all_patients,c88e1fb267a4aed17d3779cab916445229089304,32
method_comparison_all_genes,c88e1fb267a4aed17d3779cab916445229089304,157
method_comparison_clinical_only,c88e1fb267a4aed17d3779cab916445229089304,8
method_comparison_clinical_only_unknown_values_replaced,c88e1fb267a4aed17d3779cab916445229089304,8
